# Notebook 03 — ControlNet Generation & Quality Gate

Adds ControlNet Canny conditioning to the same 10 triples from Notebook 02:
- **Naive + ControlNet** (Cell B): minimal prompt with breed shape conditioning
- **Structured + ControlNet** (Cell D): full structured prompt with breed shape conditioning

Displays the conditioning images before generation so you can verify quality,
then renders a full 4-condition comparison grid (A → B → C → D).

> **Run on Colab (T4/V100/A100).** Requires Notebook 02 outputs in `outputs/A/` and `outputs/C/`.

## 0 — Colab Bootstrap

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/<YOUR_USERNAME>/stable-diffusion.git'  # ← update
    REPO_DIR = '/content/stable-diffusion'

    if not os.path.isdir(REPO_DIR):
        !git clone $REPO_URL $REPO_DIR
    else:
        !git -C $REPO_DIR pull

    %cd $REPO_DIR
    !pip install -q -r requirements.txt
else:
    root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if os.path.basename(root) == 'stable-diffusion':
        os.chdir(root)
    print(f'Working dir: {os.getcwd()}')

## 1 — GPU Check & Imports

In [ ]:
import torch

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif device == 'cpu':
    print('WARNING: No GPU detected. ControlNet generation will be very slow.')

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml
from PIL import Image
from tqdm.auto import tqdm

from src.data.dataset import OxfordPetDataset
from src.data.masks import trimap_to_seg_map, image_to_canny
from src.data.taxonomy import load_taxonomy, species_for_breed, all_conditions, all_environments
from src.generation.seeds import get_seeds
from src.generation.io import save_image_with_sidecar
from src.pipelines.controlnet import run_controlnet
from src.pipelines.loader import load_controlnet_pipeline
from src.prompts.mapper import structured_input_to_prompt, naive_input_to_prompt

print('All imports OK')

## 2 — Same 10 Triples as Notebook 02

**Keep this list identical to notebook 02** so the ablation cells share the same (breed, condition, environment) × seed combos.

In [ ]:
TRIPLES = [
    ('beagle',              'cone_collar',    'clinic'),
    ('Siamese',             'health_exam',    'exam_room'),
    ('golden_retriever',    'bandaged_paw',   'home'),
    ('Persian',             'grooming',       'grooming_salon'),
    ('pug',                 'weight_check',   'clinic'),
    ('Maine_Coon',          'dental_check',   'exam_room'),
    ('samoyed',             'post_bath_drying', 'home'),
    ('British_Shorthair',   'vaccination',    'mobile_clinic'),
    ('boxer',               'cone_collar',    'outdoor'),
    ('Ragdoll',             'health_exam',    'home'),
]

SEEDS     = get_seeds()   # [42, 137, 2024, 9999]
OUT_ROOT  = Path('outputs')
DATA_ROOT = 'data'

print(f'{len(TRIPLES)} triples | {len(SEEDS)} seeds | {len(TRIPLES)*len(SEEDS)*2} total images (B+D)')

## 3 — Load Dataset & Build Per-Breed Index

We need one real Oxford image per breed to supply the seg map / Canny conditioning.

In [ ]:
dataset = OxfordPetDataset(root=DATA_ROOT)

# Build a mapping from breed name → first dataset index for that breed
breed_to_idx: dict[str, int] = {}
for i in range(len(dataset)):
    _, _, breed, _ = dataset[i]
    if breed not in breed_to_idx:
        breed_to_idx[breed] = i

print(f'Dataset: {len(dataset)} images, {len(breed_to_idx)} breeds')

# Check all triple breeds are in dataset (fall back to closest if absent)
missing = [b for b, _, _ in TRIPLES if b not in breed_to_idx]
if missing:
    print(f'WARNING: breeds not in dataset (will use beagle as proxy): {missing}')
    FALLBACK_BREED = 'beagle'
else:
    print('All breeds found in dataset ✓')

## 4 — Quality Gate: Inspect Seg Maps Before Generation

Oxford trimaps are single-class (pet foreground only). This cell visualises the conditioning images **before** any expensive generation so you can decide:
- If seg maps look reasonable → keep `primary: seg` in `generation.yaml`
- If seg maps look too coarse / blurry → switch to `canny` below and re-run

In [ ]:
N_PREVIEW = min(5, len(TRIPLES))  # show first 5 triples

fig, axes = plt.subplots(N_PREVIEW, 3, figsize=(12, 3.5 * N_PREVIEW))
fig.patch.set_facecolor('#0d0d0d')

axes[0][0].set_title('Source image', color='white', fontsize=9)
axes[0][1].set_title('Seg map (ADE20K animal #126)', color='white', fontsize=9)
axes[0][2].set_title('Canny edges (fallback)', color='white', fontsize=9)

for row, (breed, cond, env) in enumerate(TRIPLES[:N_PREVIEW]):
    ds_idx = breed_to_idx.get(breed, breed_to_idx.get('beagle'))
    src_img, trimap, ds_breed, _ = dataset[ds_idx]

    seg_map   = trimap_to_seg_map(trimap)
    canny_map = image_to_canny(src_img)

    for ax, img, lbl in zip(
        axes[row],
        [src_img, seg_map, canny_map],
        [f'{breed}\n({cond})', 'seg map', 'canny'],
    ):
        ax.imshow(img)
        ax.axis('off')
        ax.set_ylabel(lbl, color='white', fontsize=7, rotation=0,
                      ha='right', va='center', labelpad=55)

fig.suptitle('Control conditioning preview (seg vs canny)', color='white', fontsize=12)
plt.tight_layout(pad=0.5)
plt.savefig(OUT_ROOT / 'phase4_conditioning_preview.png', dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Preview saved → outputs/phase4_conditioning_preview.png')

### ⚠️ Quality Gate Decision

**Review the preview above.** Then set `CONTROLNET_TYPE` accordingly:

```
'seg'   → seg maps look good (distinct foreground blob, correct extent)
'canny' → seg maps are too coarse/messy; Canny edges carry more structure
```

This updates `generation.yaml` **in-place** so every downstream script inherits the decision.

In [ ]:
# ── SET THIS AFTER REVIEWING THE PREVIEW ABOVE ─────────────────────────────
CONTROLNET_TYPE = 'seg'   # change to 'canny' if seg quality is weak
# ────────────────────────────────────────────────────────────────────────────

GEN_CONFIG_PATH = 'configs/generation.yaml'

with open(GEN_CONFIG_PATH) as f:
    gen_cfg = yaml.safe_load(f)

old_primary = gen_cfg['controlnet']['primary']
gen_cfg['controlnet']['primary'] = CONTROLNET_TYPE

with open(GEN_CONFIG_PATH, 'w') as f:
    yaml.dump(gen_cfg, f, default_flow_style=False, sort_keys=False)

if old_primary != CONTROLNET_TYPE:
    print(f'⚡ Quality gate: switched primary from "{old_primary}" → "{CONTROLNET_TYPE}"')
    print('   Update TRACKER.md with this decision after the session.')
else:
    print(f'✓ Primary ControlNet unchanged: "{CONTROLNET_TYPE}"')

## 5 — Load ControlNet Pipeline

In [ ]:
cn_pipe = load_controlnet_pipeline(CONTROLNET_TYPE, GEN_CONFIG_PATH)
print(f'ControlNet-{CONTROLNET_TYPE} pipeline loaded on {next(cn_pipe.unet.parameters()).device}')

## 6 — Generate Images: With ControlNet Conditioning

**10 triples × 4 seeds × 2 prompt strategies = 80 images.**
On T4 (~30 steps, 512×512) ≈ 10–14 min total.

In [ ]:
results: dict[str, list[dict]] = {'B': [], 'D': []}
ctrl_images: list[Image.Image] = []  # keep one per triple for grid display

total = len(TRIPLES) * len(SEEDS) * 2
pbar  = tqdm(total=total, desc='Generating B+D')

for triple_idx, (breed, cond, env) in enumerate(TRIPLES):
    sp     = species_for_breed(breed)
    ds_idx = breed_to_idx.get(breed, breed_to_idx.get('beagle'))
    src_img, trimap, _, _ = dataset[ds_idx]

    naive_pp  = naive_input_to_prompt(breed=breed, condition=cond)
    struct_pp = structured_input_to_prompt(
        breed=breed, species=sp, condition=cond, environment=env
    )

    # Store control image once per triple (for display in §7)
    if CONTROLNET_TYPE == 'seg':
        ctrl_img = trimap_to_seg_map(trimap)
    else:
        ctrl_img = image_to_canny(src_img)
    ctrl_images.append(ctrl_img)

    for seed in SEEDS:
        breed_slug = breed.lower().replace(' ', '_')
        stem_base  = f'{breed_slug}__{cond}__{env}__{seed}'

        # ── Cell B ──────────────────────────────────────────────────────────
        img_b, meta_b, _ = run_controlnet(
            naive_pp, src_img,
            seed=seed, breed=breed, species=sp,
            condition=cond, environment=env, cell='B',
            trimap=trimap, controlnet_type=CONTROLNET_TYPE,
            gen_config_path=GEN_CONFIG_PATH,
        )
        save_image_with_sidecar(img_b, OUT_ROOT / 'B', f'B__{stem_base}', meta_b)
        results['B'].append({'image': img_b, 'meta': meta_b})
        pbar.update(1)

        # ── Cell D ──────────────────────────────────────────────────────────
        img_d, meta_d, _ = run_controlnet(
            struct_pp, src_img,
            seed=seed, breed=breed, species=sp,
            condition=cond, environment=env, cell='D',
            trimap=trimap, controlnet_type=CONTROLNET_TYPE,
            gen_config_path=GEN_CONFIG_PATH,
        )
        save_image_with_sidecar(img_d, OUT_ROOT / 'D', f'D__{stem_base}', meta_d)
        results['D'].append({'image': img_d, 'meta': meta_d})
        pbar.update(1)

pbar.close()
print(f"Cell B: {len(list((OUT_ROOT/'B').glob('*.png')))} images")
print(f"Cell D: {len(list((OUT_ROOT/'D').glob('*.png')))} images")

## 7 — Full 4-Cell Comparison Grid (seed=42)

Columns: **Control image | Cell A (naive) | Cell B (naive+CN) | Cell C (struct) | Cell D (struct+CN)**  
Each row = one triple.  
This is the core visual evidence for the slide deck.

In [ ]:
# Load Cell A and C images from disk (generated in notebook 02)
def _load_cell_images(cell: str, triples, seeds) -> list[Image.Image]:
    """Load the first-seed image for each triple from outputs/<cell>/."""
    imgs = []
    seed0 = seeds[0]
    for breed, cond, env in triples:
        slug = breed.lower().replace(' ', '_')
        stem = f'{cell}__{slug}__{cond}__{env}__{seed0}'
        p = OUT_ROOT / cell / f'{stem}.png'
        if p.exists():
            imgs.append(Image.open(p))
        else:
            # Fallback: blank
            imgs.append(Image.new('RGB', (512, 512), color=(40, 40, 40)))
            print(f'  WARNING: {p} not found — run notebook 02 first')
    return imgs

imgs_a = _load_cell_images('A', TRIPLES, SEEDS)
imgs_c = _load_cell_images('C', TRIPLES, SEEDS)

# Cell B and D: take seed=0 (first) from results
N_SEEDS = len(SEEDS)
imgs_b = [results['B'][i * N_SEEDS]['image'] for i in range(len(TRIPLES))]
imgs_d = [results['D'][i * N_SEEDS]['image'] for i in range(len(TRIPLES))]

N_ROWS = len(TRIPLES)
N_COLS = 5  # ctrl | A | B | C | D

COL_LBLS  = ['Control', 'A\nNaive', 'B\nNaive+CN', 'C\nStructured', 'D\nStruct+CN']
COL_CLRS  = ['#2a2a2a', '#1a3050', '#1a2840', '#1a4030', '#103020']

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(N_COLS * 3, N_ROWS * 3.2))
fig.patch.set_facecolor('#0d0d0d')

for col_i, lbl in enumerate(COL_LBLS):
    axes[0][col_i].set_title(lbl, color='white', fontsize=9, pad=4)

for row_i, (breed, cond, env) in enumerate(TRIPLES):
    sp    = species_for_breed(breed)
    row_lbl = f'{breed.replace("_"," ")}\n{cond}'

    row_imgs = [ctrl_images[row_i], imgs_a[row_i], imgs_b[row_i], imgs_c[row_i], imgs_d[row_i]]
    for col_i, img in enumerate(row_imgs):
        ax = axes[row_i][col_i]
        ax.imshow(img)
        ax.axis('off')
        ax.set_facecolor(COL_CLRS[col_i])

    axes[row_i][0].set_ylabel(row_lbl, color='white', fontsize=7,
                               rotation=0, ha='right', va='center', labelpad=70)

patches = [
    mpatches.Patch(color=COL_CLRS[i], label=COL_LBLS[i].replace('\n', ' '))
    for i in range(N_COLS)
]
fig.legend(handles=patches, loc='upper center', ncol=N_COLS,
           fontsize=9, facecolor='#1a1a1a', edgecolor='gray', labelcolor='white',
           bbox_to_anchor=(0.5, 1.01))

fig.suptitle(
    f'2×2 Ablation — All 4 Cells (seed={SEEDS[0]}, ControlNet={CONTROLNET_TYPE})',
    color='white', fontsize=13, y=1.04,
)

plt.tight_layout(pad=0.5)
grid_path = OUT_ROOT / 'phase4_full_ablation_grid.png'
plt.savefig(grid_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Grid saved → {grid_path}')

## 8 — Sanity Checks

In [ ]:
# Verify output counts and spot-check one sidecar
for cell in ('A', 'B', 'C', 'D'):
    pngs  = list((OUT_ROOT / cell).glob('*.png'))
    jsons = list((OUT_ROOT / cell).glob('*.json'))
    ok    = '✓' if len(pngs) == len(jsons) else '✗ MISMATCH'
    print(f'Cell {cell}: {len(pngs)} images, {len(jsons)} sidecars  {ok}')

print()

# Check Cell D sidecar for ControlNet fields
d_json = sorted((OUT_ROOT / 'D').glob('*.json'))[0]
d_meta = json.loads(d_json.read_text())
print('Cell D sidecar fields:')
for k, v in d_meta.items():
    print(f'  {k:35s} {str(v)[:70]}')

## 9 — Qualitative Notes

Fill after reviewing the grid above. Used in Phase 5 evaluation and slide deck.

---

### ControlNet quality gate outcome
- **Modality chosen:** `seg` / `canny` _(circle one)_
- **Reason:** _(e.g. "seg maps were clean — distinct foreground blob, no artefacts")_

### Cell B vs Cell A (ControlNet adds shape, same naive prompt)

| Triple | Improvement? | Notes |
|--------|-------------|-------|
| beagle / cone_collar / clinic | | |
| Siamese / health_exam / exam_room | | |
| pug / weight_check / clinic | | |

### Cell D vs Cell C (ControlNet adds shape, structured prompt)

| Triple | Improvement? | Notes |
|--------|-------------|-------|
| beagle / cone_collar / clinic | | |
| Siamese / health_exam / exam_room | | |
| pug / weight_check / clinic | | |

### Failure cases (anatomy errors, missing conditions, wrong environments)
- _(list here — feed into Phase 5 failure analysis)_

### Overall: does Cell D dominate?
_(Expected: yes — structured prompt + ControlNet should be clearest and most condition-faithful)_

---
**End of Notebook 03.**
Next → `04_evaluation.ipynb` (quantitative evaluation).